# Minnesota Food Shelf Exploratory Data Analysis (EDA)

This notebook consolidates data from several sources and presents visualizations that summarize the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Merging and Cleaning

This section prepares the data for analysis by combining the population information from `bea_mn_2024.csv` and the food shelf list from `food_shelves_scrapedgeocoded_2026.pkl` into a single `county` table.

### Cleaning the County Population Data

`bea_mn_2024.csv`.

In [ ]:
bea = pd.read_csv("data/bea_mn_2024.csv", header=3)
len(bea)

The BEA table has footnotes that need to be removed.

In [ ]:
# Displaying the footnotes:
with pd.option_context("display.max_colwidth", None):
    display(bea.loc[:, ["GeoFIPS"]].tail(5))

In [ ]:
# Dropping the footnotes:
bea = bea.drop([264, 265, 266, 267, 268])
# Checking the tail:
bea.tail(3)

The BEA table has summary rows for the whole state that need to be removed.

In [ ]:
# Displaying the summary rows:
bea.head(3)

In [ ]:
# Dropping the MN total rows:
bea = bea.drop([0, 1, 2])
# Checking the head:
bea.head(3)

We can pivot the table so that each row is a county.

In [ ]:
# Pivoting the table so that each row is a county:

# Mapping LineCode to column names:
linecode_map = {1.0: "income", 2.0: "population", 3.0: "income_per_capita"}

# Pivoting and renaming columns:
county = (
    bea[bea["LineCode"].isin(linecode_map.keys())]
    .assign(variable=lambda df: df["LineCode"].map(linecode_map))
    .pivot_table(index="GeoName", columns="variable", values="2024", aggfunc="first")
    .reset_index()
    .rename(columns={"GeoName": "county"})
    .rename_axis(None, axis=1)
)[["county", "income", "population", "income_per_capita"]]

county.head(3)

We perform some common sense checks to verify the data are internally consistent.

In [ ]:
# Checking the number of counties correctly equals 87:
print(f"Total number of counties: {len(county)}\n")

# Checking the sums given by the BEA:
# Total MN population according to the BEA: 5793151.0
print(f"Total MN population (in thousands): {county['population'].sum()}")
# Total personal MN income according to the BEA: 437981871.0
print(f"Total MN personal income (thousands of $): {county['income'].sum()}\n")

# Checking the per capita calculation:
aitkin_inc = county.at[0, "income"]
aitkin_pop = county.at[0, "population"]
aitkin_cap = county.at[0, "income_per_capita"]
print(f"Aitkin County income: {aitkin_inc}")
print(f"Aitkin County population: {aitkin_pop}")
print(f"Aitkin County income per capita (from BEA): {aitkin_cap}")
print(
    f"Aitkin County income per capita (calculated): {round((aitkin_inc * 1000.0) / aitkin_pop,0)}"
)

We standardize the county names.

In [ ]:
# Standardizing county names:
county["county"] = county["county"].str.replace(", MN", "")
county[["county"]].sample(5, random_state=42).sort_values("county")

We check if there are non-integer `income` or `population` values, and then cast to int:

In [ ]:
# Checking for non-whole values:
print(
    county.loc[county["income"] % 1 != 0, "income"].count(),
    county.loc[county["population"] % 1 != 0, "population"].count(),
    county.loc[county["income_per_capita"] % 1 != 0, "income_per_capita"].count(),
)

In [ ]:
# Casting to int:
county["income"] = county["income"].astype("int")
county["population"] = county["population"].astype("int")
county["income_per_capita"] = county["income_per_capita"].astype("int")

# Example row:
county.loc[county["county"] == "Ramsey"]

### Cleaning the Food Shelf Data

The data is from `food_shelves_scrapedgeocoded_2026.pkl`. See `food_shelf_list_reconciliation.ipynb` for some preliminary comparisons to other lists.

In [ ]:
shelf = pd.read_pickle("data/food_shelves_scrapedgeocoded_2026.pkl")
len(shelf)

In [ ]:
# Manually adding missing food shelves:

# Adding Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul
# Adding Second Harvest Northland: Koochiching County
new_rows = pd.DataFrame(
    [
        {
            "name": "Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul",
            "address": "797 E 7th St, Saint Paul, MN 55106",
            "lat": pd.NA,
            "lng": pd.NA,
            "county": "Ramsey County",
        },
        {
            "name": "Second Harvest Northland: Koochiching County",
            "address": "10370 MN-11, Birchdale, MN, USA",
            "lat": pd.NA,
            "lng": pd.NA,
            "county": "Koochiching County",
        },
    ]
)
shelf = pd.concat([shelf, new_rows], ignore_index=True)
len(shelf)

In [ ]:
# Checking for missing or mangled counties:
shelf.loc[shelf["county"].isna() | (shelf["county"].str.len() < 10), "county"].count()

We need to make sure we're only including Minnesota counties in our analysis.

In [ ]:
# Display non-Minnesota food shelves:
(
    shelf.loc[~shelf["address"].str.contains("MN")]
    .loc[~shelf["address"].str.contains("Mn")]
    .loc[~shelf["address"].str.contains("Minnesota")]
    .loc[:, ["name", "address", "county"]]
)

Three of these are manged Minnesota addresses, but the rest are in Wisconsin and North Dakota, and can be dropped.

In [ ]:
# Dropping all non-Minnesota food shelves:
mn_addresses = shelf.iloc[[66, 101, 540]]["address"]
non_mn_idx = (
    shelf.loc[~shelf["address"].str.contains("MN")]
    .loc[~shelf["address"].str.contains("Mn")]
    .loc[~shelf["address"].str.contains("Minnesota")]
    .loc[:, ["name", "address", "county"]]
    .loc[~shelf["address"].isin(mn_addresses)]
    .index
)
shelf = shelf.drop(index=non_mn_idx)

len(shelf)

We standardize the county names.

In [ ]:
shelf["county"] = shelf["county"].str.replace("County", "")
shelf["county"] = shelf["county"].str.strip()

print(f"Unique county names: {shelf['county'].nunique()}")

It's not a problem that there's fewer than 87 counties represented, as a county can have 0 food shelves.

However, we do need to compare our county lists to make sure they will merge correctly:

In [1]:
bea_counties = county["county"].unique()
shelf_counties = shelf["county"].unique()

for c in bea_counties:
    if c not in shelf_counties:
        print(f"{c} is unmatched.")

NameError: name 'county' is not defined

<img src="https://github.com/a-location-1/mn-food-shelf-analysis/blob/main/images/wilkincountyfoodshelf.png" alt='A screenshot of the Food Group's Find Help Map showing the Richland Wilkin Food Pantry' width=60%>

So, Wilkin county is served by a food shelf, but it's in ND. So the solution is to instead re-include that food shelf. But we don't want to unnecessarily bias our per capita, so lets also add in the population and income for the nieghboring North Dakota county. 

In [ ]:
# copy-pasted, to potentially play with
shelf.loc[shelf["name"].str.contains("Salvation Army")]

In [ ]:
# copy-pasted not fixed
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(shelf.loc[shelf["match"] != "Match", "address"])

In [ ]:
metro_counties = [
    "Anoka",
    "Carver",
    "Dakota",
    "Hennepin",
    "Ramsey",
    "Scott",
    "Washington",
]